# 04 - Ringkasan Hasil

Tabel 3, plot komparatif F1, analisis segmentation failures, feature importance.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.evaluate import aggregate_feature_importance, plot_feature_importance, print_summary_table
from src.utils import get_project_paths, read_best_enhancement

paths = get_project_paths()
metrics_dir = paths["metrics"]


## Tabel 3 - Ringkasan Semua Skenario

In [ ]:
summary = print_summary_table(metrics_dir)
summary


## Plot Komparatif F1-Score

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=summary, x="scenario_id", y="f1_weighted", hue="model", ax=ax)
    ax.set_title("F1-Score (weighted) per Skenario")
    ax.set_xlabel("Skenario")
    plt.tight_layout()
    plt.savefig(paths["figures"] / "f1_comparison.png", dpi=150)
    plt.show()


## Enhancement Terpilih (E*)

In [ ]:
print(f"Best enhancement: {read_best_enhancement(metrics_dir)}")


## Segmentation Failures

In [ ]:
fail_path = metrics_dir / "segmentation_failures.csv"
if fail_path.exists():
    fails = pd.read_csv(fail_path)
    display(fails.groupby("commodity").size().sort_values(ascending=False).head(10))
else:
    print("Belum ada log segmentasi. Jalankan skenario dengan segmentasi aktif.")


## Uji Signifikansi McNemar

In [ ]:
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
else:
    print("Jalankan notebook 02 dan 03 untuk uji McNemar.")


## Feature Importance (Skenario 9 - RF)

In [ ]:
import joblib

rf_path = paths["models"] / "rf_scenario_09.pkl"
if rf_path.exists():
    rf = joblib.load(rf_path)
    from src.features import get_feature_group_names
    names = get_feature_group_names("all", segmented=True)
    if len(names) != len(rf.feature_importances_):
        names = [f"f{i}" for i in range(len(rf.feature_importances_))]
    labels, vals = aggregate_feature_importance(rf.feature_importances_, names)
    plot_feature_importance(vals, labels, save_path=paths["figures"] / "feature_importance_s09.png")
    plt.show()
else:
    print("Model RF S9 belum tersedia. Jalankan notebook 02 terlebih dahulu.")


## Inference Time Comparison

In [ ]:
if not summary.empty and "inference_time_ms_per_image" in summary.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=summary, x="scenario_id", y="inference_time_ms_per_image", ax=ax)
    ax.set_title("Inference Time (ms/image)")
    plt.tight_layout()
    plt.show()


## Kelemahan Sistem & Limitasi

Bagian ini wajib dibahas dalam laporan (rubrik #9). Bukan kegagalan sistem,
melainkan batas kebenaran klaim yang jujur harus diakui.

### 1. Single split, satu seed
Seluruh angka F1/akurasi bertumpu pada **satu pembagian data acak** (SEED=42,
70/15/15). Tidak ada repeated runs atau confidence interval. Akibatnya:
- Perbedaan F1 kecil antar skenario (mis. 0.92 vs 0.89) bisa jadi noise split.
- McNemar membantu di level prediksi per-sampel, tapi tidak menangkap varians
  yang muncul kalau seed diganti.
- **Saran pengembangan**: k-fold lintas seed, atau setidaknya 3 seed berbeda.

### 2. Risiko leakage near-duplicate
Dataset Kaggle buah/sayur sering memuat banyak foto dari **objek fisik yang sama**
dalam kondisi pencahayaan/sudut berbeda. Split acak per-citra bisa menaruh
near-duplicate di train **dan** test sekaligus, menggelembungkan semua angka.
- Tidak ada deteksi duplikat yang dilakukan (membutuhkan image hashing atau
  perceptual similarity).
- Angka performa yang sangat tinggi (>95% F1) harus dibaca dengan hati-hati
  karena kemungkinan mengandung kontaminasi ini.
- **Saran pengembangan**: deduplikasi dengan pHash sebelum split.

### 3. Desain ladder, bukan factorial
Setiap skenario mengubah satu variabel terhadap sibling-nya (*one-factor-at-a-time*).
Ini memudahkan interpretasi tapi **tidak bisa menangkap interaksi antar faktor**:
- Belum diuji: apakah CLAHE efektif *tanpa* SSR, atau manfaatnya bergantung SSR?
- Belum diuji: apakah segmentasi membantu pada komoditas tertentu tapi menyakiti
  yang lain?
- **Saran pengembangan**: desain 2x2 factorial (SSR on/off x segmentasi on/off)
  untuk melihat interaksi.

### 4. Segmentasi berbasis threshold (Otsu) tanpa learning
Metode segmentasi - Otsu pada kanal S (HSV) + grayscale + morfologi - sederhana
dan cepat, tapi gagal pada buah/sayur gelap berlatar belakang serupa (tercatat
di `segmentation_failures.csv`). Fallback (gambar penuh) dipakai saat foreground
<5%, artinya fitur segmentasi kehilangan maknanya pada sebagian sampel.


## Per-Commodity Performance Comparison

In [ ]:
s5_pred_path = metrics_dir / "predictions_s5.csv"
s10_pred_path = metrics_dir / "predictions_s10.csv"
s11_pred_path = metrics_dir / "predictions_s11.csv"

dfs_comm = []
from sklearn.metrics import f1_score
label_map = {"fresh": 0, "rotten": 1}

if s5_pred_path.exists():
    s5_preds = pd.read_csv(s5_pred_path)
    s5_preds["true_encoded"] = s5_preds["label"].map(label_map)
    s5_comm = []
    for comm, group in s5_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s5_comm.append({"commodity": comm, "samples": len(group), "f1_s5": f1})
    dfs_comm.append(pd.DataFrame(s5_comm))

if s10_pred_path.exists():
    s10_preds = pd.read_csv(s10_pred_path)
    s10_preds["true_encoded"] = s10_preds["label"].map(label_map)
    s10_comm = []
    for comm, group in s10_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s10_comm.append({"commodity": comm, "f1_s10": f1})
    dfs_comm.append(pd.DataFrame(s10_comm))

if s11_pred_path.exists():
    s11_preds = pd.read_csv(s11_pred_path)
    s11_preds["true_encoded"] = s11_preds["label"].map(label_map)
    s11_comm = []
    for comm, group in s11_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s11_comm.append({"commodity": comm, "f1_s11": f1})
    dfs_comm.append(pd.DataFrame(s11_comm))

if len(dfs_comm) > 0:
    df_compare = dfs_comm[0]
    for df in dfs_comm[1:]:
        df_compare = pd.merge(df_compare, df, on="commodity")
    
    # Sort by the best available model's F1
    sort_col = "f1_s11" if "f1_s11" in df_compare.columns else ("f1_s10" if "f1_s10" in df_compare.columns else "f1_s5")
    df_compare = df_compare.sort_values(sort_col, ascending=False)
    display(df_compare)
    
    value_vars = [c for c in ["f1_s5", "f1_s10", "f1_s11"] if c in df_compare.columns]
    df_melted = df_compare.melt(id_vars=["commodity", "samples"], value_vars=value_vars,
                                var_name="model", value_name="f1_score")
    df_melted["model"] = df_melted["model"].map({"f1_s5": "S5 SVM", "f1_s10": "S10 CNN (Pipeline)", "f1_s11": "S11 CNN (Raw)"})
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(data=df_melted, x="commodity", y="f1_score", hue="model", ax=ax)
    ax.set_title("F1-Score per Komoditas (Perbandingan Model)")
    ax.set_ylabel("Weighted F1-Score")
    ax.set_xlabel("Komoditas")
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(paths["figures"] / "commodity_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Tidak ada data prediksi S5, S10, atau S11 yang ditemukan.")
